# Batch Processing with All Agents

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))


from agents.prompt_generator_agent import PromptGeneratorAgent
from agents.schema_generator_agent import SchemaGeneratorAgent
from agents.ExtractionAgent import TranscriptExtractionAgent
from utils.bootstrap_extraction import ExtractionBootstrapEvaluator
from agents.JudgeAgent import JudgeAgent
from spectrum_client import *

# Import Libraries
import pandas as pd
import json

# from sql.load_databricks import DatabricksConnector
import csv
import os

In [2]:
prompt = """Using the internal KNOWLEDE in the file VAOs , generate a prompt to identify if there was a VAO by the agent on the call, dont include any explanation fields Include VAO language examples for potential transcript searching. use examples from the docment INCLUDE some a quote and reasoning column"""

In [3]:
# prompt = """ what was the callers primary reason for calling the department"""

In [4]:
prompt_generator_agent = PromptGeneratorAgent(model='gpt-4.1-mini',temperature=0.4,rag_folder=fr'knowledge_base')

result = prompt_generator_agent.run(user_input=prompt)

[PromptGeneratorAgent] Initialized with 1 tool(s)
[PromptGeneratorAgent] Using rag_folder: C:\Users\P3311043\Python Projects\Transcript_Analysis_Automation\knowledge_base
=== OUTGOING PAYLOAD ===
{
  "model": "gpt-4.1-mini",
  "input": "system: \nClassify the user's message into one of two intents:\n- \"generate\": the user wants a new prompt created or an existing prompt modified/updated\n- \"converse\": the user is asking a question, giving feedback, or discussing without requesting a change\n\nReply with JSON only: {\"intent\": \"generate\"} or {\"intent\": \"converse\"}\n\nuser: Using the internal KNOWLEDE in the file VAOs , generate a prompt to identify if there was a VAO by the agent on the call, dont include any explanation fields Include VAO language examples for potential transcript searching. use examples from the docment INCLUDE some a quote and reasoning column",
  "messages": [
    {
      "role": "system",
      "content": "\nClassify the user's message into one of two in

In [5]:
result

PromptModel(system_prompt='You are an expert analyst tasked with identifying whether a Value Addition Opportunity (VAO) was presented by the agent during a customer call. Use the internal knowledge provided about VAOs to guide your identification. Focus only on detecting the presence of a VAO, quoting relevant transcript excerpts, and providing concise reasoning based on the VAO language examples. Do not include any explanation fields beyond what is requested.', user_prompt='You will be given a transcript of a call between an agent and a customer. Using the internal VAO knowledge and examples provided below, determine if the agent presented a VAO during the call.\n\nVAO Language Examples for reference:\n- VAOs provide opportunities for customers to increase the value of their services by recommending additional services or packages.\n- VAOs are best presented during the Communicate Solutions and Value section of the call.\n- Use positive framing to reinforce benefits.\n- VAO offers may

In [6]:

system_prompt = result.system_prompt
user_prompt = result.user_prompt
output_format = result.output_format

In [7]:
print(system_prompt)
print(user_prompt)
print(output_format)

You are an expert analyst tasked with identifying whether a Value Addition Opportunity (VAO) was presented by the agent during a customer call. Use the internal knowledge provided about VAOs to guide your identification. Focus only on detecting the presence of a VAO, quoting relevant transcript excerpts, and providing concise reasoning based on the VAO language examples. Do not include any explanation fields beyond what is requested.
You will be given a transcript of a call between an agent and a customer. Using the internal VAO knowledge and examples provided below, determine if the agent presented a VAO during the call.

VAO Language Examples for reference:
- VAOs provide opportunities for customers to increase the value of their services by recommending additional services or packages.
- VAOs are best presented during the Communicate Solutions and Value section of the call.
- Use positive framing to reinforce benefits.
- VAO offers may include keywords such as Keep, Switch, Add to d

In [8]:
schema_agent = SchemaGeneratorAgent(model='gpt-4.1-mini',temperature=0.4)

schema = schema_agent.run(output_format,user_prompt,fr'data_models\generated')

[SchemaGeneratorAgent] Initialized with 0 tool(s)
[SchemaGeneratorAgent] Prepared 0 tool schema(s)
=== TOOL SCHEMAS ===
[]
[SchemaGeneratorAgent] Prepared 0 available function(s)
[SchemaGeneratorAgent] Calling chat_with_tools with 2 message(s), 0 tool(s), tool_choice=auto
=== OUTGOING PAYLOAD ===
{
  "model": "gpt-4.1-mini",
  "input": "system: \nYou are an expert Python engineer. Given a set of output field definitions from an extraction prompt,\nproduce a Pydantic v2 BaseModel class that captures those fields with the most appropriate Python types.\n\nType selection rules:\n- Use Literal[\"a\", \"b\", ...] for any categorical, enum-like, or fixed-choice fields.\n- Use bool for yes/no or true/false fields.\n- Use int or float for numeric fields.\n- Use str only for genuinely free-text fields (e.g. summaries, verbatim quotes).\n- Wrap any field that may not always be present in Optional[...] = None.\n- Every field must have a Field(description=\"...\") matching the original field descr

In [9]:
import importlib.util
from pathlib import Path

def load_model_from_path(file_path, class_name):
    file_path = Path(file_path)

    spec = importlib.util.spec_from_file_location(file_path.stem, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return getattr(module, class_name)

In [10]:
result, path = schema  # or however you're unpacking it

ModelClass = load_model_from_path(path, result.model_name)

print(ModelClass)
# <class 'reason_for_call.ReasonForCall'>

<class 'vao_presentation.VaoPresentation'>


In [11]:
# Load GPT
# connect to client
gpt = SpectrumClient(model='gpt-4.1-mini',temperature=0.1)

In [12]:
# Read Data
input_file = fr'TEST_DATA.csv'
df = pd.read_csv(input_file,dtype={'UCID':str})

In [ ]:


output_file = 'test_data_output_n.csv'

def get_template_params(row):
    """
    Maps the row values to the Jinja template parameters for the prompt.
    """
    return {

    }

extraction_agent = TranscriptExtractionAgent(
    model="gpt-4.1-mini",
    temperature="0.1",
)

#  try:
#             self.extraction_result = pd.read_csv(self.extraction_results_path,dtype={'UCID':str})
#         except:
#             print(f'[ERROR]: Error finding extraction results using the file path: {self.extraction_result }\n switching to locally saved dataframe results')
results = extraction_agent.process_batch(
        transcript_column_name='TRANSCRIPT',
        rows=df.iloc[:,:].to_dict('records'),
        response_format=ModelClass,
        template_params={},
        system_prompt=system_prompt,
        prompt_template=user_prompt,
        max_workers=3,
        token_threshold=5,
        output_file=output_file,
        include_columns= ['TRANSCRIPT']
    )

[TranscriptExtractionAgent] Initialized with 0 tool(s)


NameError: name 'ModelClass' is not defined

In [14]:
results = pd.read_csv(output_file)

In [15]:
results

,AGENTRECORDINGSESSIONID,PROCESS_STATUS,TRANSCRIPT,VAO_present,VAO_quote,VAO_reasoning,RETRIEVAL_FOUND_ANY,USED_MATCHED_EVIDENCE,TRANSCRIPT_TOKENS,EVIDENCE_CHUNK,CHUNK_TOKENS,ATTEMPTED_SEARCH_TERMS,LLM_INPUT_TOKENS,LLM_OUTPUT_TOKENS,LLM_TOTAL_TOKENS
0,401174911E85D845923BD28FF9A6FAD2,True,Agent: 5.56 SECONDS - Hello this is Dominique ...,False,NaN,The agent focused solely on troubleshooting th...,False,False,4681,NaN,0,recommend | suggest | benefit | save | discoun...,5114,78,5192
1,BA0A435AF32B064C94BD59F5A6109266,True,Agent: 4.12 SECONDS - Thank you for calling Sp...,False,NaN,The agent focused solely on addressing the cus...,False,False,698,NaN,0,customer solutions team | remove TV package | ...,1131,76,1207
2,697F58DEA46F9840BF22D8CF4622526D,True,Agent: 2.76 SECONDS - Hi thank you for calling...,True,Looking into your things i am pulling a few st...,The agent presented a Value Addition Opportuni...,False,False,6091,NaN,0,free of charge | help you set that up | call y...,6524,170,6694
3,AC4B025C39646245B39C82CFCDABA12A,True,Agent: 7.80 SECONDS - Good afternoon and thank...,True,Now umm just to recap you called in because yo...,The agent presented a value addition opportuni...,False,False,3517,NaN,0,manage your account | self service tips | my s...,3950,161,4111
4,927081B1D104E74CB677239B6FC553F2,True,Agent: 5.40 SECONDS - Thank you for calling Sp...,True,That's kind of where I was going with it. That...,The agent presented a value addition opportuni...,False,False,9281,NaN,0,modem for free | modem free | free modem | add...,9714,118,9832


In [16]:
results.columns

Index(['AGENTRECORDINGSESSIONID', 'PROCESS_STATUS', 'TRANSCRIPT',
       'VAO_present', 'VAO_quote', 'VAO_reasoning', 'RETRIEVAL_FOUND_ANY',
       'USED_MATCHED_EVIDENCE', 'TRANSCRIPT_TOKENS', 'EVIDENCE_CHUNK',
       'CHUNK_TOKENS', 'ATTEMPTED_SEARCH_TERMS', 'LLM_INPUT_TOKENS',
       'LLM_OUTPUT_TOKENS', 'LLM_TOTAL_TOKENS'],
      dtype='str')

In [17]:
row = results.iloc[3].to_dict()
print("VAO_present raw:", row.get("VAO_presented"), type(row.get("VAO_presented")))
print("VAO_quote raw:", row.get("VAO_quote"), type(row.get("VAO_quote")))
print("VAO_reasoning raw:", row.get("VAO_reasoning"), type(row.get("VAO_reasoning")))

VAO_present raw: None <class 'NoneType'>
VAO_quote raw: Now umm just to recap you called in because you were trying to uh troubleshoot and make sure that you we have received your payment always remember you can manage your account and get some great self service tips by visiting us at my spectrum app this is just a suggestion you don't have to do it i know you don't totally Trust being online so it it's always option if you would like. <class 'str'>
VAO_reasoning raw: The agent presented a value addition opportunity by suggesting the customer use the My Spectrum app to manage their account and access self-service tips, which is an additional service option to increase the value of the customer's service. This recommendation was positively framed as a helpful suggestion and presented after addressing the customer's concerns. <class 'str'>


In [24]:
judge_agent = JudgeAgent()
eval = 'test_data_output_eval.csv'
judge_config = [
        {
            'field_name':'VAO_present',
            'claim_column':'VAO_present',
            'quote_column':'VAO_quote',
            'reasoning_column':'VAO_reasoning',
            'task_prompt':f"{system_prompt}"
        }
]

row = results.iloc[0].to_dict()
field_inputs = judge_agent.build_judge_field_inputs(row, judge_config)

# print(field_inputs)

# judge_results = judge_agent.run(
#     transcript=row["TRANSCRIPT"],
#     field_inputs=field_inputs,
#     context_lines=2,
# )

# print("JUDGE_RESULTS:", judge_results)

judge_results = judge_agent.process_batch(
    transcript_column_name='TRANSCRIPT',
    rows=results.iloc[:,:].to_dict('records'),
    judge_config=judge_config,
    output_file=eval,
    include_columns=['VAO_present','TRANSCRIPT']
)

[JudgeAgent] Initialized with 0 tool(s)
Judging Sessioon ID: 401174911E85D845923BD28FF9A6FAD2
Judging Sessioon ID: BA0A435AF32B064C94BD59F5A6109266
Judging Sessioon ID: 697F58DEA46F9840BF22D8CF4622526D
[JudgeAgent] Initialized with 0 tool(s)
FIELD_INPUTS: [JudgeFieldInput(task_prompt='You are an expert analyst tasked with identifying whether a Value Addition Opportunity (VAO) was presented by the agent during a customer call. Use the internal knowledge provided about VAOs to guide your identification. Focus only on detecting the presence of a VAO, quoting relevant transcript excerpts, and providing concise reasoning based on the VAO language examples. Do not include any explanation fields beyond what is requested.', field_name='VAO_present', claim_value=False, quote='', ai_reasoning="The agent focused solely on troubleshooting the customer's internet connectivity issue without recommending any additional services, upgrades, or packages. There was no positive framing or mention of keepi

In [34]:
from utils.bootstrap_extraction import ExtractionBootstrapEvaluator
import utils.bootstrap_extraction
importlib.reload(utils.bootstrap_extraction)

judge_df = pd.read_csv('test_data_output_eval.csv')
judge_df
evaluator = ExtractionBootstrapEvaluator(judge_df.iloc[:].to_dict('records'),
                                         TranscriptExtractionAgent)

evaluation_result, repeated_runs_df = evaluator.evaluate(
    n_runs=3,
    judge_df=judge_df,
    run_consistency=True,
    prompt_template=user_prompt,
    response_format=ModelClass,
    consistency_sample_size=5,
    template_params={},   # dict or callable
    system_prompt=system_prompt,
    fields=["VAO_present"],
    max_workers=10,
    token_threshold=10,
)

evaluator.write_results_to_csv(
    evaluation_result=evaluation_result,
    repeated_runs_df=repeated_runs_df,
    summary_output_file="evaluation_summary.csv",
    repeated_runs_output_file="repeated_runs.csv",
)

Starting extraction run 1/3
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: BA0A435AF32B064C94BD59F5A6109266
Processing Session ID: 401174911E85D845923BD28FF9A6FAD2
Processing Session ID: 697F58DEA46F9840BF22D8CF4622526D
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 927081B1D104E74CB677239B6FC553F2
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Registered tool: search_transcript
[TranscriptExtractionAgent] Tool mode iteration 1/10
Processing Session ID: AC4B025C39646245B39C82CFCDABA12A
[TranscriptExtractionAgent] Registered tool: search_transcript
[TranscriptExtractionAgent] Tool mode iteration 1/10
[TranscriptExtractionAgent] Prepared 1 tool schema(s)
=== TOOL SCHEMAS ===
[
  {
    "type": "function",
    "name": "search_transcript",
    "description": "Search the transcript for one or more relevant terms and return matching transcript lines with line numbers and surrounding context. Also

In [35]:
evaluation_result.consistency_quality

ConsistencyQuality(consistency_rate=1.0, consistency_count=5, total_evaluated=5)

In [36]:
repeated_runs_df

,AGENTRECORDINGSESSIONID,PROCESS_STATUS,VAO_present,VAO_quote,VAO_reasoning,RETRIEVAL_FOUND_ANY,USED_MATCHED_EVIDENCE,TRANSCRIPT_TOKENS,EVIDENCE_CHUNK,CHUNK_TOKENS,ATTEMPTED_SEARCH_TERMS,LLM_INPUT_TOKENS,LLM_OUTPUT_TOKENS,LLM_TOTAL_TOKENS,RUN_ID,CONSISTENCY_SAMPLE_SIZE
0,401174911E85D845923BD28FF9A6FAD2,TRUE,False,,The agent focused solely on troubleshooting th...,False,False,4681,,0,would you like to add | can I offer you | swit...,5114,87,5201,1,5
1,AC4B025C39646245B39C82CFCDABA12A,TRUE,True,always remember you can manage your account an...,The agent presented a value addition opportuni...,False,False,3517,,0,manage your account | self service tips | my s...,3950,119,4069,1,5
2,BA0A435AF32B064C94BD59F5A6109266,TRUE,False,,The agent focused solely on addressing the cus...,False,False,698,,0,connect you to our customer solutions team | r...,1131,76,1207,1,5
3,927081B1D104E74CB677239B6FC553F2,TRUE,True,That's why I wanted to come back over and kind...,The agent presented a value addition opportuni...,False,False,9281,,0,recommend upgrading | recommend adding | recom...,9714,160,9874,1,5
4,697F58DEA46F9840BF22D8CF4622526D,TRUE,True,Looking into your things i am pulling a few st...,The agent presented a Value Addition Opportuni...,False,False,6091,,0,free of charge | help you set up | call you ba...,6524,173,6697,1,5
5,BA0A435AF32B064C94BD59F5A6109266,TRUE,False,,The agent focused on addressing the customer's...,False,False,698,,0,customer solutions team | remove TV package | ...,1131,97,1228,2,5
6,401174911E85D845923BD28FF9A6FAD2,TRUE,False,,The agent focused solely on troubleshooting th...,False,False,4681,,0,Would you like to add | How about upgrading | ...,5114,78,5192,2,5
7,AC4B025C39646245B39C82CFCDABA12A,TRUE,True,always remember you can manage your account an...,The agent presented an opportunity for the cus...,False,False,3517,,0,value | benefit | recommend | additional | ser...,3950,130,4080,2,5
8,927081B1D104E74CB677239B6FC553F2,TRUE,True,That's kind of where I was going with it. That...,The agent presented a value addition opportuni...,False,False,9281,,0,VAO | value add | value addition | additional ...,9714,121,9835,2,5
9,697F58DEA46F9840BF22D8CF4622526D,TRUE,True,Looking into your things i am pulling a few st...,The agent presented a value addition opportuni...,False,False,6091,,0,free of charge | help you set that up | call y...,6524,171,6695,2,5


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))

import importlib.util
import orchestration.orchestrator
import agents.prompt_generator_agent
import agents.ExtractionAgent
import agents.schema_generator_agent
import agents.JudgeAgent
import pandas as pd

importlib.reload(orchestration.orchestrator)
importlib.reload(agents.prompt_generator_agent)
importlib.reload(agents.ExtractionAgent)
importlib.reload(agents.schema_generator_agent)
importlib.reload(agents.JudgeAgent)


from orchestration.orchestrator import Orchestrator
from agents.prompt_generator_agent import PromptGeneratorAgent
from agents.schema_generator_agent import SchemaGeneratorAgent
from agents.ExtractionAgent import TranscriptExtractionAgent
from agents.JudgeAgent import JudgeAgent






prompt_agent = PromptGeneratorAgent(model='gpt-4.1-mini',temperature=0.4,rag_folder=fr'knowledge_base')
schema_agent = SchemaGeneratorAgent(model='gpt-4.1-mini',temperature=0.4)
extraction_agent = TranscriptExtractionAgent(
    model="gpt-4.1-mini",
    temperature="0.1",
)

judge_agent = JudgeAgent()
prompt = """Using the internal KNOWLEDE in the file VAOs , generate a prompt to identify if there was a VAO by the agent on the call, dont include any explanation fields Include VAO language examples for potential transcript searching. use examples from the docment INCLUDE some a quote and reasoning column"""


# Read Data
input_file = fr'TEST_DATA.csv'
df = pd.read_csv(input_file,dtype={'UCID':str})

orchestrator = Orchestrator(
    df=df,
    prompt=prompt,
    PromptAgent=prompt_agent,
    SchemaAgent=schema_agent,
    TranscriptExtractionAgent=extraction_agent,
    JudgeAgent=judge_agent,
    output_dir=fr"\Users\P3311043\Python Projects\Transcript_Analysis_Automation\tests",
    project_name='VAO'
)

result = orchestrator.run(
    review_prompt=False,
    review_schema=False,
    run_judging=True,
    run_evaluation=True,
    n_runs=3,
    min_correctness_rate=1.95,
    min_consistency_rate=0.95,
    max_hallucination_rate=0.1,
    judge_kwargs={
        "max_workers": 5,
        'template_params':{}
    },
    extraction_kwargs={
        "consistency_sample_size": 50,
        "random_seed": 42,
        'template_params':{},
        "max_workers": 5,
        "token_threshold": 5,
    },
    
    
)

[PromptGeneratorAgent] Initialized with 1 tool(s)
[PromptGeneratorAgent] Using rag_folder: C:\Users\P3311043\Python Projects\Transcript_Analysis_Automation\knowledge_base
[SchemaGeneratorAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[JudgeAgent] Initialized with 0 tool(s)
[PromptGeneratorAgent] Prepared 1 tool schema(s)
[PromptGeneratorAgent] Prepared 1 available function(s)
[PromptGeneratorAgent] Calling chat_with_tools with 2 message(s), 1 tool(s), tool_choice=auto
[PromptGeneratorAgent] Executing tool: search_documents
[PromptGeneratorAgent] Tool input: {'queries': ['VAO'], 'context_lines': 3, 'regex_mode': False}
[PromptGeneratorAgent] Validated input for tool: search_documents
[PromptGeneratorAgent] Tool result from search_documents: {"found_any": true, "queries": ["VAO"], "results_by_query": [{"query": "VAO", "found": true, "match_count": 10, "matches": [{"file": "C:\\Users\\P3311043\\Python Projects\\Transcript_Analysis_Automation\\know

Assessment FAILED:
  - correctness_rate (100.00%) < threshold (195.00%)


[TranscriptExtractionAgent] Unregistered tool: search_transcript
Completed 5/5: AGENTRECORDINGSESSIONID 697F58DEA46F9840BF22D8CF4622526D | Status: TRUE
[PromptGeneratorAgent] Prepared 1 tool schema(s)
[PromptGeneratorAgent] Prepared 1 available function(s)
[PromptGeneratorAgent] Calling chat_with_tools with 6 message(s), 1 tool(s), tool_choice=auto
Starting extraction run 1/1
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 697F58DEA46F9840BF22D8CF4622526D
Processing Session ID: 401174911E85D845923BD28FF9A6FAD2
Processing Session ID: BA0A435AF32B064C94BD59F5A6109266
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: AC4B025C39646245B39C82CFCDABA12A
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 927081B1D104E74CB677239B6FC553F2
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Registered tool: search_transcript
[TranscriptExtractionAgent] Tool mode iteration 1/10
[Tran

Assessment FAILED:
  - correctness_rate (100.00%) < threshold (195.00%)
  - consistency_rate (80.00%) < threshold (95.00%)


[TranscriptExtractionAgent] Unregistered tool: search_transcript
Completed 5/5: AGENTRECORDINGSESSIONID 697F58DEA46F9840BF22D8CF4622526D | Status: TRUE
[PromptGeneratorAgent] Prepared 1 tool schema(s)
[PromptGeneratorAgent] Prepared 1 available function(s)
[PromptGeneratorAgent] Calling chat_with_tools with 8 message(s), 1 tool(s), tool_choice=auto
Starting extraction run 1/1
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 697F58DEA46F9840BF22D8CF4622526D
Processing Session ID: 401174911E85D845923BD28FF9A6FAD2
Processing Session ID: BA0A435AF32B064C94BD59F5A6109266
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: AC4B025C39646245B39C82CFCDABA12A
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 927081B1D104E74CB677239B6FC553F2
[TranscriptExtractionAgent] Registered tool: search_transcript
[TranscriptExtractionAgent] Tool mode iteration 1/10
[Tran

Assessment FAILED:
  - correctness_rate (100.00%) < threshold (195.00%)
  - consistency_rate (60.00%) < threshold (95.00%)


[TranscriptExtractionAgent] Unregistered tool: search_transcript
Completed 5/5: AGENTRECORDINGSESSIONID 697F58DEA46F9840BF22D8CF4622526D | Status: TRUE
[PromptGeneratorAgent] Prepared 1 tool schema(s)
[PromptGeneratorAgent] Prepared 1 available function(s)
[PromptGeneratorAgent] Calling chat_with_tools with 10 message(s), 1 tool(s), tool_choice=auto
Starting extraction run 1/1
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: 697F58DEA46F9840BF22D8CF4622526D
Processing Session ID: 401174911E85D845923BD28FF9A6FAD2
Processing Session ID: BA0A435AF32B064C94BD59F5A6109266
[TranscriptExtractionAgent] Initialized with 0 tool(s)
Processing Session ID: AC4B025C39646245B39C82CFCDABA12A
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Initialized with 0 tool(s)
[TranscriptExtractionAgent] Registered tool: search_transcript
[TranscriptExtractionAgent] Tool mode iteration 1/10
[TranscriptExtractionAgent] Prepared 1 tool schema(s)
[Trans

Assessment FAILED:
  - correctness_rate (100.00%) < threshold (195.00%)
Pipeline still FAILING after 3 revision cycles — max_revision_cycles exhausted.


[TranscriptExtractionAgent] Unregistered tool: search_transcript
Completed 5/5: AGENTRECORDINGSESSIONID 927081B1D104E74CB677239B6FC553F2 | Status: TRUE


In [9]:
result['judge_aggregation']

JudgeAggregation(id=UUID('a7f77de8-c089-4781-a7a8-55696022c527'), total_rows=5, total_field_judgments=5, total_failures=0, total_severe_failures=0, overall_grounded_rate=1.0, overall_hallucination_rate=0.0, overall_failure_rate=0.0, field_summary=FieldSummary(worst_by_hallucination_rate=[], worst_by_ungrounded_rate=[], worst_by_severe_failures=[]), error_summaries=[], representative_examples=[], prompt_lessons=PromptLessons(keep_doing=['Fields with high grounding rates (VAO_claim_field) — the extraction approach for these fields is working well.', 'Evidence retrieval is succeeding for most fields — continue requiring supporting quotes.'], stop_doing=[], prompt_changes=[]), created_at=datetime.datetime(2026, 8, 18, 19, 49, 45, 670300, tzinfo=datetime.timezone.utc))

In [2]:
for item in orchestrator.prompt_management:
    print( item)

('id', UUID('2025de73-ccef-48a2-9f25-e81f66340a79'))
('prompt_title', 'VAO')
('version', 1)
('parent_version_id', None)
('inital_user_prompt_request', 'Using the internal KNOWLEDE in the file VAOs , generate a prompt to identify if there was a VAO by the agent on the call, dont include any explanation fields Include VAO language examples for potential transcript searching. use examples from the docment INCLUDE some a quote and reasoning column')
('generated_system_prompt', 'You are an expert at analyzing call center transcripts to identify if the agent presented a Value Addition Opportunity (VAO) during the call. Use the internal knowledge about VAOs to guide your analysis. Focus on detecting VAO language examples and phrases from the call. Do not provide explanations or additional commentary beyond the requested fields.')
('generated_user_prompt', 'You will be given a transcript of a call center interaction between an agent and a customer. Your task is to determine if the agent presen